# CDM Connector Test for Azure Synapse Analytics

This notebook demonstrates how to read and write Common Data Model (CDM) entities using the Spark 3.5 CDM Connector in Azure Synapse Analytics Spark 3.5.

## Prerequisites
- Upload `spark-cdm-connector-assembly-spark3.5-1.20.0.jar` to your Synapse Analytics environment.
- Upload the `test-cdm-data` folder an ADLS Gen2 storage account. Important: keep the subfolder structure.
- Refer to the [Installation Guide](/documentation/INSTALLATION_GUIDE.md) for detailed instructions on setting up your workspace and Spark environment.

## Step 1: Configure the CDM Connector with CDM folder path and service principal information

In [ ]:
# Specify configuration for your environment
# Replace these values with your actual Fabric environment or Azure Synapse details:

# 1. OneLake Storage Configuration
ONELAKE_STORAGE = "onelake.dfs.fabric.microsoft.com"    # Do not change
WORKSPACE_NAME = "YOUR_WORKSPACE_NAME"                  # Replace with your workspace name
LAKEHOUSE_NAME = "YOUR_LAKEHOUSE_NAME"                  # Replace with your lakehouse name
LAKEHOUSE_CDM_FOLDER = "YOUR_CDM_FOLDER"                # Replace with your path to the cdm folder in your lakehouse

# 2. ADLS Gen2 Configuration
ADLS_STORAGE = "YOURADLSACCOUNT.dfs.core.windows.net"   # Replace with your ADLS Gen2 storage account name
ADLS_CONTAINER_NAME = "YOUR_CONTAINER_NAME"             # Replace with your storage container
ADLS_CDM_PATH = "YOUR_ADLS_CDM_PATH"                    # Replace with your ADLS Gen2 path to the CDM folder

# 2. Authentication Configuration
# 2a. Azure Synapse Managed Identity (recommended for Azure Synapse)
#     The Synapse Managed Identity must have access to OneLake and/or ADLS Gen2 storage accounts.
#     No additional configuration is needed here.

# 2b. Service Principal Authentication
#     Recommended for ADLS Gen2. Service principal requires "Storage Blob Data Reader" or "Storage Blob Data Contributor" role on the ADLS Gen2 storage account or container.
APP_ID = "your-service-principal-app-id"      # Your service principal client ID
APP_KEY = "your-service-principal-secret"     # Your service principal secret
TENANT_ID = "your-tenant-id"                  # Your Azure AD tenant ID
# 2b. SAS Token Authentication
#     Optional for accessing ADLS Gen2. Not supported for OneLake.
#     If using service principal authentication, this can be omitted.
ADLS_SAS_KEY = "your-adls-sas-token"          # Your ADLS Gen2 SAS token for the storage account or container.


# 3. Manifest Path Construction
MANIFEST_NAME = f"your-manifest-name.manifest.cdm.json" # Replace with your CDM manifest file name, e.g., "CdmSampleData.manifest.cdm.json"

ADLS_MANIFEST_PATH = f"{ADLS_CONTAINER_NAME}/{ADLS_CDM_PATH}/{MANIFEST_NAME}"
ONELAKE_MANIFEST_PATH = f"{WORKSPACE_NAME}/{LAKEHOUSE_NAME}.Lakehouse/Files/{LAKEHOUSE_CDM_FOLDER}/{MANIFEST_NAME}"

# 4. Print Configuration
print ("Your configuration:")
print(f"OneLake Storage:            {ONELAKE_STORAGE}")
print(f"OneLake Manifest Path:      {ONELAKE_MANIFEST_PATH}")
print(f"ADLS Storage:               {ADLS_STORAGE}")
print(f"ADLS Manifest Path:         {ADLS_MANIFEST_PATH}")
print(f"Service Principal App Id:   {APP_ID}")
print(f"Service Principal Key:      {APP_KEY}")
print(f"Tenant Id:                  {TENANT_ID}")
print(f"ADLS SAS Key:               {ADLS_SAS_KEY}")

## Step 2: Test CDM Connector - Read Employee Entity (Mixed CSV + Parquet)

In [ ]:
# Test reading the Employee CDM entity into a DataFrame
print("📖 Testing CDM Connector - Reading Employee entity...")

employees = spark.read \
    .format("com.microsoft.cdm") \
    .option("storage", ADLS_STORAGE) \
    .option("manifestPath", ADLS_MANIFEST_PATH) \
    .option("entity", "Employee") \
    .load()

# Optionally add authentication options to spark.read() if using service principal or SAS token. Default is Synapse Managed Identity.
#    .option("appId", APP_ID) \
#    .option("appKey", APP_KEY) \
#    .option("tenantId", TENANT_ID) \
# or
#    .option("sasToken", ADLS_SAS_KEY) \

# Display data and schema
print("✅ Employee entity read successfully!")
print("\nSchema:")
employees.printSchema()
display(employees)

# Perform some analytics
print("\n📊 Employee Analytics:")
active_employees = employees.filter(employees.IsActive == True).count()
avg_salary = employees.agg({"Salary": "avg"}).collect()[0][0]

print(f"   Active Employees: {active_employees:0}")
print(f"   Average Salery: {avg_salary:,.2f}")

## Step 3: Test CDM Connector - Read SalesOrder Entity (Parquet only)

In [ ]:
import pyspark.sql.functions as F

# Test reading the SalesOrder entity
print("📖 Testing CDM Connector - Reading SalesOrder entity...")

sales_orders = spark.read \
    .format("com.microsoft.cdm") \
    .option("storage", ADLS_STORAGE) \
    .option("manifestPath", ADLS_MANIFEST_PATH) \
    .option("entity", "SalesOrder") \
    .load()

# Optionally add authentication options to spark.read() if using service principal or SAS token. Default is Synapse Managed Identity.
#    .option("appId", APP_ID) \
#    .option("appKey", APP_KEY) \
#    .option("tenantId", TENANT_ID) \
# or
#    .option("sasToken", ADLS_SAS_KEY) \

print("✅ SalesOrder entity read successfully!")
print("\nSchema:")
sales_orders.printSchema()
display(sales_orders)

# Some analytics
print("\n📊 Quick Analytics:")
print(f"\nTotal sales order records: {sales_orders.count()}")

total_revenue = sales_orders.agg(F.sum("OrderTotal").alias("total_revenue")).collect()[0][0]
avg_order_value = sales_orders.agg(F.avg("OrderTotal").alias("avg_order")).collect()[0][0]
total_freight = sales_orders.agg(F.sum("Freight").alias("total_freight")).collect()[0][0]

print(f"   Total Revenue: ${total_revenue:,.2f}")
print(f"   Average Order Value: ${avg_order_value:,.2f}")
print(f"   Total Freight Costs: ${total_freight:,.2f}")

print("\n🌍 Orders by Country:")

display(sales_orders.groupBy("ShipCountry").agg(
    F.count("*").alias("order_count"),
    F.sum("OrderTotal").alias("revenue")
).orderBy(F.desc("revenue")))

## Step 4: Read Customer Entity (csv file) and test Cross-Entity Joins

In [ ]:
# Test joining data across CDM entities
print("🔗 Testing Cross-Entity Joins...")

print("📖 Reading Customer entity...")
# Read Customer entity
customers = spark.read \
    .format("com.microsoft.cdm") \
    .option("storage", ADLS_STORAGE) \
    .option("manifestPath", ADLS_MANIFEST_PATH) \
    .option("entity", "Customer") \
    .load()

# Optionally add authentication options to spark.read() if using service principal or SAS token. Default is Synapse Managed Identity.
#    .option("appId", APP_ID) \
#    .option("appKey", APP_KEY) \
#    .option("tenantId", TENANT_ID) \
# or
#    .option("sasToken", ADLS_SAS_KEY) \

print("✅ Customer entity read successfully!")
print("\nSchema:")
customers.printSchema()
display(customers)

# Join sales orders with employees and customers
print("🔗  Joining sales orders with employees and customers...")
order_details = sales_orders \
    .join(employees, sales_orders.EmployeeId == employees.EmployeeId, "left") \
    .join(customers, sales_orders.CustomerId == customers.CustomerId, "left") \
    .select(
        sales_orders.OrderId,
        customers.CustomerName,
        F.concat(employees.FirstName, F.lit(" "), employees.LastName).alias("SalesRep"),
        sales_orders.OrderDate,
        sales_orders.OrderTotal,
        sales_orders.ShipCountry
    )

print("\n🎯 Order Details with Customer and Employee Information:")
display(order_details)

print("\n👥 Top Sales Representatives:")
display(order_details.groupBy("SalesRep").agg(
    F.count("*").alias("orders_count"),
    F.sum("OrderTotal").alias("total_sales")
).orderBy(F.desc("total_sales")))